librerías y ruta

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path

PROYECTO_DIR = Path.cwd().resolve() 
DATOS_DIR = PROYECTO_DIR / "Datos"

Lista de activos y descarga de 2004-2025

In [17]:
ACTIVOS = [
    # Renta variable US
    "^GSPC",   # S&P 500 - large cap blend
    "^NDX",    # Nasdaq 100 - growth/tech
    "IWM",     # Russell 2000 - small cap
    "XLF",     # Financieras US
    "XLE",     # Energía US

    # Renta variable internacional
    "EEM",     # Emerging Markets
    "EWJ",     # Japón
    "FEZ",     # Euro Stoxx 50

    # Renta fija
    "SHY",     # Bonos corto plazo (1-3 años)
    "AGG",     # Aggregate bonds (intermedio)
    "TLT",     # Bonos largo plazo (20+ años)
    "TIP",     # Bonos ligados a inflación
    "LQD",     # High yield bonds

    # Commodities
    "GC=F",    # Oro
    "HG=F",    # Cobre 

    # Real estate
    "VNQ",     # REITs US

    # Risk-free proxy
    "^IRX",    # T-Bill yield (no es activo invertible, solo para rf)
]

start = "2004-01-01"
end   = "2026-01-01"

data_completa_2004_2025 = yf.download(ACTIVOS, start=start, end=end, auto_adjust=False, progress=False)

precios = data_completa_2004_2025["Adj Close"].copy()
precios.index = pd.to_datetime(precios.index).normalize()
precios = precios.sort_index()
precios = precios[~precios.index.duplicated(keep="first")]

precios.head(), precios.tail()

(Ticker            AGG        EEM        EWJ        FEZ        GC=F    HG=F  \
 Date                                                                         
 2004-01-02  50.081432  12.069787  27.436586  17.436970         NaN     NaN   
 2004-01-05  50.140598  12.493290  28.002279  17.768866  424.399994  1.0830   
 2004-01-06  50.367451  12.352118  27.747723  17.862988  422.799988  1.0670   
 2004-01-07  50.480835  12.380930  27.521437  17.615299  421.899994  1.0625   
 2004-01-08  50.465988  12.427744  27.747723  17.967010  424.000000  1.1045   
 
 Ticker            IWM        LQD        SHY        TIP        TLT  VNQ  \
 Date                                                                     
 2004-01-02  41.800541  44.696507  54.080872  50.895241  40.582226  NaN   
 2004-01-05  42.289501  44.745396  54.087440  50.825024  40.476479  NaN   
 2004-01-06  42.233517  45.148441  54.153206  51.150959  40.923454  NaN   
 2004-01-07  42.681435  45.185139  54.192646  51.191082  41.086857  Na

In [18]:
print("Filas:", len(precios))
print("Columnas:", precios.shape[1])
print("Rango fechas:", precios.index.min().date(), "->", precios.index.max().date())

print("\n% NaN por activo:")
nan_pct = precios.isna().mean() * 100
display(nan_pct.sort_values(ascending=False).round(2))

print("\nPrimera fecha válida por activo:")
display(precios.apply(lambda s: s.first_valid_index()).to_frame("first_valid"))

print("\nÚltima fecha válida por activo:")
display(precios.apply(lambda s: s.last_valid_index()).to_frame("last_valid"))

print("\nMovimiento diario máximo (%) por activo:")
ret_temp = precios.pct_change(fill_method=None)
display((ret_temp.abs().max() * 100).sort_values(ascending=False).round(2).to_frame("max_abs_move_%"))

Filas: 5542
Columnas: 17
Rango fechas: 2004-01-02 -> 2025-12-31

% NaN por activo:


Ticker
VNQ      3.48
GC=F     0.27
^IRX     0.22
HG=F     0.20
AGG      0.13
TLT      0.13
^GSPC    0.13
XLF      0.13
XLE      0.13
SHY      0.13
TIP      0.13
EEM      0.13
LQD      0.13
IWM      0.13
FEZ      0.13
EWJ      0.13
^NDX     0.13
dtype: float64


Primera fecha válida por activo:


,first_valid
Ticker,
AGG,2004-01-02
EEM,2004-01-02
EWJ,2004-01-02
FEZ,2004-01-02
GC=F,2004-01-05
HG=F,2004-01-05
IWM,2004-01-02
LQD,2004-01-02
SHY,2004-01-02



Última fecha válida por activo:


,last_valid
Ticker,
AGG,2025-12-31
EEM,2025-12-31
EWJ,2025-12-31
FEZ,2025-12-31
GC=F,2025-12-31
HG=F,2025-12-31
IWM,2025-12-31
LQD,2025-12-31
SHY,2025-12-31



Movimiento diario máximo (%) por activo:


,max_abs_move_%
Ticker,
^IRX,1214.29
EEM,22.77
HG=F,22.25
XLE,20.14
VNQ,19.51
FEZ,17.53
XLF,16.67
EWJ,15.82
IWM,13.27


In [19]:
first_valid = precios.apply(lambda s: s.first_valid_index())

fecha_inicio_real = first_valid.max()
print("Fecha inicio común:", fecha_inicio_real)

last_valid = precios.apply(lambda s: s.last_valid_index())

fecha_fin_real = last_valid.min()
print("Fecha fin común:", fecha_fin_real)

precios = precios.loc[(precios.index >= fecha_inicio_real) & (precios.index <= fecha_fin_real)]

Fecha inicio común: 2004-09-29 00:00:00
Fecha fin común: 2025-12-31 00:00:00


In [20]:
print("Total NaN por activo:")
display((precios.isna().sum()).sort_values(ascending=False))

print("\n% NaN por activo:")
display((precios.isna().mean()*100).round(3).sort_values(ascending=False))

print("Filas con algún NaN:", precios.isna().any(axis=1).sum())

nan_blocks = precios.isna().astype(int)

for col in precios.columns:
    max_block = nan_blocks[col].groupby(
        (nan_blocks[col] != nan_blocks[col].shift()).cumsum()
    ).sum().max()
    print(f"{col}: bloque máximo NaN = {max_block}")

Total NaN por activo:


Ticker
GC=F     14
^IRX     12
HG=F     10
AGG       7
TLT       7
^GSPC     7
XLF       7
XLE       7
VNQ       7
SHY       7
TIP       7
EEM       7
LQD       7
IWM       7
FEZ       7
EWJ       7
^NDX      7
dtype: int64


% NaN por activo:


Ticker
GC=F     0.261
^IRX     0.224
HG=F     0.187
AGG      0.131
TLT      0.131
^GSPC    0.131
XLF      0.131
XLE      0.131
VNQ      0.131
SHY      0.131
TIP      0.131
EEM      0.131
LQD      0.131
IWM      0.131
FEZ      0.131
EWJ      0.131
^NDX     0.131
dtype: float64

Filas con algún NaN: 23
AGG: bloque máximo NaN = 2
EEM: bloque máximo NaN = 2
EWJ: bloque máximo NaN = 2
FEZ: bloque máximo NaN = 2
GC=F: bloque máximo NaN = 1
HG=F: bloque máximo NaN = 1
IWM: bloque máximo NaN = 2
LQD: bloque máximo NaN = 2
SHY: bloque máximo NaN = 2
TIP: bloque máximo NaN = 2
TLT: bloque máximo NaN = 2
VNQ: bloque máximo NaN = 2
XLE: bloque máximo NaN = 2
XLF: bloque máximo NaN = 2
^GSPC: bloque máximo NaN = 2
^IRX: bloque máximo NaN = 2
^NDX: bloque máximo NaN = 2


Se han seleccionado activos que permitan formar una cartera diversificada y además que no tengan grandes bloques de NaNs, los NaN que hay ahora se deben a cierres de mercado con días/horas de diferencia que se desajustan, pero se puede solventar fácil, un bloque grande de NaN sería más perjudicial, implicaría que para un periodo se han perdido los datos.

In [21]:
precios = precios.sort_index()
precios = precios.ffill()

print("Total NaN por activo:")
display((precios.isna().sum()).sort_values(ascending=False))

print("\n% NaN por activo:")
display((precios.isna().mean()*100).round(3).sort_values(ascending=False))

print("Filas con algún NaN:", precios.isna().any(axis=1).sum())

print("ALgún activo con precio negativo?\n", (precios <= 0).sum())


Total NaN por activo:


Ticker
AGG      0
TIP      0
^IRX     0
^GSPC    0
XLF      0
XLE      0
VNQ      0
TLT      0
SHY      0
EEM      0
LQD      0
IWM      0
HG=F     0
GC=F     0
FEZ      0
EWJ      0
^NDX     0
dtype: int64


% NaN por activo:


Ticker
AGG      0.0
TIP      0.0
^IRX     0.0
^GSPC    0.0
XLF      0.0
XLE      0.0
VNQ      0.0
TLT      0.0
SHY      0.0
EEM      0.0
LQD      0.0
IWM      0.0
HG=F     0.0
GC=F     0.0
FEZ      0.0
EWJ      0.0
^NDX     0.0
dtype: float64

Filas con algún NaN: 0
ALgún activo con precio negativo?
 Ticker
AGG      0
EEM      0
EWJ      0
FEZ      0
GC=F     0
HG=F     0
IWM      0
LQD      0
SHY      0
TIP      0
TLT      0
VNQ      0
XLE      0
XLF      0
^GSPC    0
^IRX     7
^NDX     0
dtype: int64


Ya no hay NaN y el único activo con "precio" negativo es IRX, pero esto no es precio es un yield anual, por lo que puede ser negativo (tipos de interés negativos) es el único yield y luego veremos que lo separamos del resto para usarlo como activo libre de riesgo rf.

In [22]:
# separar IRX
irx_yield = precios["^IRX"].copy()
precios_activos = precios.drop(columns=["^IRX"])

# retornos pct:change calcula la varación con respecto al día anterior
retornos = precios_activos.pct_change(fill_method=None)

# eliminar primera fila ya que no puede comparar con el día anterior
retornos = retornos.dropna(how="any")

In [23]:
print("Rango retornos:")
print(retornos.index.min().date(), "→", retornos.index.max().date())
print("Número de días:", len(retornos))

print("NaN totales:", retornos.isna().sum().sum())

resumen_extremos = pd.DataFrame({
    "min": retornos.min(),
    "max": retornos.max()
})

display((resumen_extremos * 100).round(2).sort_values("min"))

Rango retornos:
2004-09-30 → 2025-12-31
Número de días: 5355
NaN totales: 0


,min,max
Ticker,,
HG=F,-22.25,13.25
XLE,-20.14,16.47
VNQ,-19.51,17.01
XLF,-16.67,16.46
EEM,-16.17,22.77
IWM,-13.27,9.15
FEZ,-12.46,17.53
^NDX,-12.19,12.58
^GSPC,-11.98,11.58


No se ve ningún dato demasiado desorbitado (+400% en una semana)

Vamos a construir ahora rf

In [24]:
# IRX es un yield anual, por lo que está en porcentaje anual
rf_anual = irx_yield.loc[retornos.index] / 100.0

print("Rango rf anual:")
print(rf_anual.min(), "→", rf_anual.max())

rf_diario = (1.0 + rf_anual) ** (1.0 / 252.0) - 1.0

print("Rango rf diario:")
print(rf_diario.min(), "→", rf_diario.max())

print("NaN en rf_anual:", rf_anual.isna().sum())
print("NaN en rf_diario:", rf_diario.isna().sum())


Rango rf anual:
-0.0010499999672174453 → 0.053480000495910646
Rango rf diario:
-4.168846879260002e-06 → 0.00020676331788527236
NaN en rf_anual: 0
NaN en rf_diario: 0


Rf tiene sentido
Vamos a hacer las features
Las features son variables que se derivan de los retornos históricos, aportan información relevante par nuestro futuro modelo, como momentum(tendencia) y volatilidad(riesgo). Se usan ventanas de 20 y 60 días para calcularlas, por eso perdemos los primeros 60 días. Además se desplazan un día hacia delante para no obtener estas variables con información actual y tener por tanto data leakage.

In [25]:
def construir_features(ret):
    ret = ret.sort_index().copy()

    momentum_20 = ret.rolling(20).sum().add_suffix("_mom20")
    momentum_60 = ret.rolling(60).sum().add_suffix("_mom60")

    volatilidad_20 = ret.rolling(20).std(ddof=1).add_suffix("_vol20")
    volatilidad_60 = ret.rolling(60).std(ddof=1).add_suffix("_vol60")

    feats = pd.concat([momentum_20, momentum_60, volatilidad_20, volatilidad_60], axis=1)

    #evitar leakage
    feats = feats.shift(1)
    feats = feats.dropna()

    return feats

features = construir_features(retornos)

In [26]:
print("Rango features:", features.index.min(), "→", features.index.max())
print("Fechas iguales retornos:", features.index.equals(retornos.loc[features.index].index))
print("NaNs features:", features.isna().sum().sum())

Rango features: 2004-12-27 00:00:00 → 2025-12-31 00:00:00
Fechas iguales retornos: True
NaNs features: 0


Para enriquecer el estado del agente, se incorporan también los retornos del día anterior junto con las features (momentum y volatilidad). Esto permite al modelo disponer de información inmediata sobre el comportamiento reciente del mercado, además de las dinámicas suavizadas capturadas por las ventanas móviles. Los retornos se desplazan un día (lag 1) para evitar data leakage, garantizando que en cada instante solo se utilice información disponible antes de tomar la decisión de inversión.

In [27]:
# retornos retrasados 1 día para evitar leakage
retornos_lag1 = retornos.shift(1)
# alinear con features (features ya vienen con shift(1) + dropna)
retornos_lag1 = retornos_lag1.loc[features.index]
# estado exógeno final
datos_estado = pd.concat([features, retornos_lag1], axis=1)
datos_estado = datos_estado.dropna()

# 2) alineación final obligatoria evita data leakage
fechas = (datos_estado.index.intersection(retornos.index).intersection(rf_diario.index))

datos_estado = datos_estado.loc[fechas]
retornos_ok = retornos.loc[fechas]
rf_ok = rf_diario.loc[fechas]
rf_anual_ok = rf_anual.loc[fechas]

print("datos_estado:", datos_estado.shape)
print("retornos_ok:", retornos_ok.shape)
print("rf_ok:", rf_ok.shape)
print("rf_anual_ok:", rf_anual_ok.shape)
print("Fechas iguales:",
      datos_estado.index.equals(retornos_ok.index) and
      datos_estado.index.equals(rf_ok.index))

datos_estado: (5295, 80)
retornos_ok: (5295, 16)
rf_ok: (5295,)
rf_anual_ok: (5295,)
Fechas iguales: True


Esto es lo que queremos:
- datos_estado: (5477, 60) → 60 variables de estado exógeno (features + retornos lag1)
- retornos_ok: (5477, 12) → 12 activos riesgosos para el reward
- rf_ok: (5477,) → rf diario y anual alineados
- Fechas iguales: True → alineación sin leakage por desfase

In [28]:
fecha_fin_train = "2016-01-01"
fecha_fin_validation = "2020-01-01"

# =========================
# ESTADO
# =========================
datos_estado_train = datos_estado.loc[datos_estado.index < fecha_fin_train]

datos_estado_validation = datos_estado.loc[
    (datos_estado.index >= fecha_fin_train)&(datos_estado.index < fecha_fin_validation)]

datos_estado_test = datos_estado.loc[datos_estado.index >= fecha_fin_validation]

# =========================
# RETORNOS (reward)
# =========================
retornos_train = retornos_ok.loc[datos_estado_train.index]
retornos_validation = retornos_ok.loc[datos_estado_validation.index]
retornos_test = retornos_ok.loc[datos_estado_test.index]

# =========================
# RISK FREE
# =========================
rf_train = rf_ok.loc[datos_estado_train.index]
rf_validation = rf_ok.loc[datos_estado_validation.index]
rf_test = rf_ok.loc[datos_estado_test.index]

rf_anual_train = rf_anual_ok.loc[datos_estado_train.index]
rf_anual_validation = rf_anual_ok.loc[datos_estado_validation.index]
rf_anual_test = rf_anual_ok.loc[datos_estado_test.index]

print("Train:", datos_estado_train.shape, retornos_train.shape, rf_train.shape, rf_anual_train.shape)
print("Validation:", datos_estado_validation.shape, retornos_validation.shape, rf_validation.shape, rf_anual_validation.shape)
print("Test:", datos_estado_test.shape, retornos_test.shape, rf_test.shape, rf_anual_test.shape)

Train: (2778, 80) (2778, 16) (2778,) (2778,)
Validation: (1006, 80) (1006, 16) (1006,) (1006,)
Test: (1511, 80) (1511, 16) (1511,) (1511,)


In [29]:
from entorno_cartera import EntornoCartera

entorno_train = EntornoCartera(datos_estado_train, retornos_train, rf_diario=rf_train)

estado_inicial = entorno_train.reset()
print("Dimensión estado entorno:", estado_inicial.shape)
print("Numero activos entorno:", entorno_train.numero_activos)

TypeError: EntornoCartera.__init__() got an unexpected keyword argument 'rf_diario'

Lo guardamos todo en csv

In [20]:
CARPETA_DATOS = Path.cwd() / "Datos"   # si el notebook se ejecuta desde la raíz del proyecto
for sub in ["Raw", "Train", "Validation", "Test"]:
    (CARPETA_DATOS / sub).mkdir(parents=True, exist_ok=True)

# Guardar estado (features + retornos_lag1)
datos_estado.to_csv(CARPETA_DATOS / "Raw" / "datos_estado_full.csv")
datos_estado_train.to_csv(CARPETA_DATOS / "Train" / "datos_estado_train.csv")
datos_estado_validation.to_csv(CARPETA_DATOS / "Validation" / "datos_estado_validation.csv")
datos_estado_test.to_csv(CARPETA_DATOS / "Test" / "datos_estado_test.csv")

# Guardar retornos (reward)
retornos_ok.to_csv(CARPETA_DATOS / "Raw" / "retornos_full.csv")
retornos_train.to_csv(CARPETA_DATOS / "Train" / "retornos_train.csv")
retornos_validation.to_csv(CARPETA_DATOS / "Validation" / "retornos_validation.csv")
retornos_test.to_csv(CARPETA_DATOS / "Test" / "retornos_test.csv")

# Guardar rf diario
rf_ok.to_csv(CARPETA_DATOS / "Raw" / "rf_diario_full.csv")
rf_train.to_csv(CARPETA_DATOS / "Train" / "rf_diario_train.csv")
rf_validation.to_csv(CARPETA_DATOS / "Validation" / "rf_diario_validation.csv")
rf_test.to_csv(CARPETA_DATOS / "Test" / "rf_diario_test.csv")

rf_anual_ok.to_csv(CARPETA_DATOS / "Raw" / "rf_anual_full.csv")
rf_anual_train.to_csv(CARPETA_DATOS / "Train" / "rf_anual_train.csv")
rf_anual_validation.to_csv(CARPETA_DATOS / "Validation" / "rf_anual_validation.csv")
rf_anual_test.to_csv(CARPETA_DATOS / "Test" / "rf_anual_test.csv")

print("CSV guardados en:", CARPETA_DATOS.resolve())

CSV guardados en: C:\Users\inigo\Desktop\TFG Info\Datos
